## Deserialization

Wie entsteht aus Rohdaten, etwa einem Dictionary oder einem JSON-Text aus einer API oder Datei, eine validierte Model-Instanz? Dieser Vorgang heißt Deserialization. Pydantic bietet dafür mehrere Wege: den Aufruf des Konstruktors mit entpacktem Dictionary, die Methode `model_validate` und für JSON-Text die Methode `model_validate_json`.

**Danach ist klar:**
- wie sich ein Dictionary mit `Person(**data)` oder `model_validate` in ein Model umwandeln lässt,
- wie `model_validate_json` einen JSON-Text direkt validiert,
- welche Fehler bei fehlenden Fields auftreten, auch bei JSON-Text.

Voraussetzung: Notebook `02 - Creating a Pydantic Model`.

In [ ]:
# `BaseModel` ist die Basisklasse für alle Models, `ValidationError` wird später gebraucht, um Fehlerfälle abzufangen.
from pydantic import BaseModel, ValidationError

In [ ]:
# Dasselbe `Person`-Model wie zuvor: drei Pflicht-Fields vom Typ `str`, `str` und `int`.
class Person(BaseModel):
    first_name: str
    last_name: str
    age : int

In [ ]:
# Der bekannte Weg: Die Werte stehen als Keyword-Argumente direkt im Aufruf.
p = Person(first_name='Isaac', last_name='Newton', age=84)

Daten aus einer API, aus JSON oder aus einer Datei liegen selten als einzelne Variablen vor, sondern als Dictionary. Wie wird daraus eine `Person`-Instanz?

In [ ]:
# `data` ist ein gewöhnliches Dictionary und noch nicht validiert. Die Keys entsprechen den Fields von `Person`.
data = {
    "first_name": "Isaac",
    "last_name": "Newton",
    "age": 84
}

**Vorhersage:** `Person(**data)` steht allein in der Zelle. Was bewirkt `**`, und was gibt die Zelle aus?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** `**data` entpackt das Dictionary in Keyword-Argumente: Aus `Person(**data)` wird `Person(first_name="Isaac", last_name="Newton", age=84)`. Der Ausdruck steht allein in der Zelle, deshalb zeigt Jupyter das Ergebnis an.
</details>

In [ ]:
Person(**data)

**Vorhersage:** `model_validate` bekommt das Dictionary als einziges Argument, ohne Entpacken. Liefert das dasselbe Ergebnis wie `Person(**data)`?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** `Person.model_validate(data)` nimmt das Dictionary direkt entgegen, ohne Entpacken. Das Ergebnis ist dasselbe, weil beide Wege dieselbe Validierung durchlaufen. Zusätzlich bietet die Methode Optionen wie `strict` und `from_attributes`.

Kurz gesagt: Aus einem Dictionary entsteht mit beiden Aufrufen dieselbe validierte Instanz.
</details>

In [ ]:
p = Person.model_validate(data)
p

In [ ]:
# `missing_data` enthält nur `last_name`. `model_validate` meldet beide fehlenden Pflicht-Fields (`type=missing`)
# gemeinsam in einer `ValidationError`, die das `try`/`except` abfängt.
missing_data = {"last_name": "Newton"}

try:
    Person.model_validate(missing_data)
except ValidationError as exc:
    print(exc)

Daten aus einer API oder Datei kommen oft als JSON-Text. Muss dieser zuerst mit `json.loads` in ein Dictionary umgewandelt werden?

In [ ]:
# `data_json` ist ein String mit JSON-Text und kein Dictionary. Die dreifachen Anführungszeichen erlauben mehrere Zeilen.
data_json = '''
{
    "first_name": "Isaac",
    "last_name": "Newton",
    "age": 84
}
'''

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** `model_validate_json` parst den JSON-Text und validiert das Ergebnis in einem Schritt. Ein Umweg über `json.loads` ist nicht nötig. Das Ergebnis ist dieselbe Instanz wie zuvor.

Kurz gesagt: `model_validate` für Dictionaries, `model_validate_json` für JSON-Text.
</details>

In [ ]:
p = Person.model_validate_json(data_json)
p

**Vorhersage:** Dieser JSON-Text ist gültig, aber unvollständig (`last_name` und `age` fehlen). Wie viele Fehler meldet Pydantic, und von welchem Typ?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Der JSON-Text ist syntaktisch gültig, das Parsen gelingt. Erst die Validierung bemerkt die fehlenden Pflicht-Fields und meldet beide gemeinsam als `missing`, genau wie bei `model_validate` mit einem Dictionary.

Hinweis: Wäre der JSON-Text selbst ungültig (z. B. ein Komma am Ende), würde stattdessen `json_invalid` gemeldet, und die Fields würden gar nicht erst geprüft.
</details>

In [ ]:
missing_data_json = '''
{
    "first_name": "Isaac"
}
'''

try:
    Person.model_validate_json(missing_data_json)
except ValidationError as exc:
    print(exc)

### Zusammenfassung

- Rohdaten aus API oder Datei liegen als Dictionary oder als JSON-Text vor.
- `Person(**data)` entpackt das Dictionary in Keyword-Argumente und erzeugt eine validierte Instanz.
- `Person.model_validate(data)` nimmt das Dictionary direkt entgegen und ist der übliche Weg, unter anderem wegen Optionen wie `strict` und `from_attributes`.
- `Person.model_validate_json(text)` parst und validiert JSON-Text in einem Schritt.
- Fehlende Fields werden gesammelt als `missing` gemeldet, bei Dictionary und JSON-Text gleichermaßen. Ungültiges JSON scheitert schon vorher mit `json_invalid`.

Weiter mit `04 - Serialization`.